In [3]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# Paths
dataset_dir = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\colored_images'
output_dir = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\augmented_data'

# Set augmentation parameters
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Get the maximum number of images in any class
max_images = 1805  # Based on your class distribution, No_DR has 1805 images

# Function to perform augmentation
def augment_data(class_folder, current_count):
    class_path = os.path.join(dataset_dir, class_folder)
    output_class_path = os.path.join(output_dir, class_folder)
    
    os.makedirs(output_class_path, exist_ok=True)
    
    # Load images from the class folder
    images = os.listdir(class_path)
    
    for image in images:
        img_path = os.path.join(class_path, image)
        img = load_img(img_path)  # Load image
        img_array = img_to_array(img)  # Convert image to array
        img_array = np.expand_dims(img_array, axis=0)  # Expand dimensions to fit the model
        
        # Generate augmented images
        for i, batch in enumerate(datagen.flow(img_array, batch_size=1, save_to_dir=output_class_path,
                                               save_prefix='aug', save_format='jpg')):
            current_count += 1
            if current_count >= max_images:  # Stop when reaching the max_images
                break

# Iterate over each class folder and augment data
for class_folder in os.listdir(dataset_dir):
    augment_data(class_folder, current_count=0)

print("Data augmentation completed.")


Data augmentation completed.


In [4]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Paths
dataset_dir = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\augmented_data'
output_dir = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\aug_split_data'

# Create output directories for train, test, and validation
for split in ['train', 'test', 'val']:
    split_dir = os.path.join(output_dir, split)
    os.makedirs(split_dir, exist_ok=True)
    for class_dir in os.listdir(dataset_dir):
        os.makedirs(os.path.join(split_dir, class_dir), exist_ok=True)

# Function to split the dataset
def split_data(class_folder):
    class_path = os.path.join(dataset_dir, class_folder)
    images = os.listdir(class_path)
    
    # Split into train and remaining (test + val)
    train_images, remaining_images = train_test_split(images, train_size=0.8, random_state=42)
    
    # Split remaining into test and val
    test_images, val_images = train_test_split(remaining_images, test_size=0.5, random_state=42)
    
    # Move files to respective folders
    for image in train_images:
        shutil.copy(os.path.join(class_path, image), os.path.join(output_dir, 'train', class_folder, image))
    
    for image in test_images:
        shutil.copy(os.path.join(class_path, image), os.path.join(output_dir, 'test', class_folder, image))
    
    for image in val_images:
        shutil.copy(os.path.join(class_path, image), os.path.join(output_dir, 'val', class_folder, image))

# Iterate over class folders and split data
for class_folder in os.listdir(dataset_dir):
    split_data(class_folder)

print("Data split completed.")

Data split completed.


code for true labels csv for aug images

In [8]:
import os
import pandas as pd

# Paths
augmented_data_dir = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\augmented_data'  # Update with your path
output_csv_path = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\augmented_labels.csv'  # Path for the output CSV file

# Define a mapping from folder names to numerical labels
class_mapping = {
    'Mild': 0,
    'Moderate': 1,
    'No_DR': 2,
    'Proliferate_DR': 3,
    'Severe': 4
}

# Create a list to hold the image file names and their corresponding labels
data = []

# Iterate through each class folder in the augmented dataset
for class_folder in os.listdir(augmented_data_dir):
    class_path = os.path.join(augmented_data_dir, class_folder)
    
    if os.path.isdir(class_path) and class_folder in class_mapping:
        # Assign a label based on the class mapping
        label = class_mapping[class_folder]
        
        # List all images in the class folder
        for image_name in os.listdir(class_path):
            # Append the image file name and its label to the data list
            data.append({'id_code': image_name, 'diagnosis': label})

# Create a DataFrame and save to CSV
labels_df = pd.DataFrame(data)
labels_df.to_csv(output_csv_path, index=False)

print("CSV file with labels for augmented images created successfully.")

CSV file with labels for augmented images created successfully.


csv for test folder

In [1]:
import os
import pandas as pd

# Paths
augmented_data_dir = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\aug_split_data\test'  # Update with your path
output_csv_path = r'C:\MyFolders\Projects\major_project\classification\dataset\aptos_resized_kaggle\aug_split_data\test_true_labels.csv'  # Path for the output CSV file

# Define a mapping from folder names to numerical labels
class_mapping = {
    'Mild': 0,
    'Moderate': 1,
    'No_DR': 2,
    'Proliferate_DR': 3,
    'Severe': 4
}

# Create a list to hold the image file names and their corresponding labels
data = []

# Iterate through each class folder in the augmented dataset
for class_folder in os.listdir(augmented_data_dir):
    class_path = os.path.join(augmented_data_dir, class_folder)
    
    if os.path.isdir(class_path) and class_folder in class_mapping:
        # Assign a label based on the class mapping
        label = class_mapping[class_folder]
        
        # List all images in the class folder
        for image_name in os.listdir(class_path):
            # Append the image file name and its label to the data list
            data.append({'id_code': image_name, 'diagnosis': label})

# Create a DataFrame and save to CSV
labels_df = pd.DataFrame(data)
labels_df.to_csv(output_csv_path, index=False)

print("CSV file with labels for augmented test images created successfully.")

CSV file with labels for augmented test images created successfully.
